In [22]:
# Đọc dữ liệu từ Bronze
df_raw_sales = (
    spark.read
    .format("parquet")
    .load("Files/Bronze/mindx_raw_sales_data")
)

display(df_raw_sales.limit(10))

StatementMeta(, 07588210-2439-412a-914d-e88280c6013e, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 29175e49-22e7-4aee-8a32-0545eb4056ad)

In [23]:
# Kiểm tra kiểu dữ liệu hiện tại của các cột
df_raw_sales.printSchema()

StatementMeta(, 07588210-2439-412a-914d-e88280c6013e, 25, Finished, Available, Finished, False)

root
 |-- currency: string (nullable = true)
 |-- customer_age: string (nullable = true)
 |-- customer_info: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- discount_code: string (nullable = true)
 |-- feedback_score: double (nullable = true)
 |-- items: string (nullable = true)
 |-- location: string (nullable = true)
 |-- loyalty_points: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- shipping_cost: double (nullable = true)
 |-- total_amount: string (nullable = true)



In [24]:
# Kiểm tra NULL

from pyspark.sql.functions import col, sum as spark_sum, when

null_counts = df_raw_sales.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_raw_sales.columns
])

display(null_counts)

StatementMeta(, 07588210-2439-412a-914d-e88280c6013e, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e4645337-a54f-4882-93ba-23ea242e6c9c)

In [25]:
# Xử lí duplicate dữ liệu
# Đếm số dòng trước khi deduplicate
raw_count = df_raw_sales.count()

# Loại bỏ các dòng trùng lặp hoàn toàn
df_sales_deduplicated = df_raw_sales.dropDuplicates()

# Đếm số dòng sau khi deduplicate
deduplicated_count = df_sales_deduplicated.count()

# Tính số dòng bị loại bỏ
removed_count = raw_count - deduplicated_count

print(f"Số dòng ban đầu: {raw_count}")
print(f"Số dòng sau khi deduplicate: {deduplicated_count}")
print(f"Số dòng trùng bị loại bỏ: {removed_count}")

StatementMeta(, 07588210-2439-412a-914d-e88280c6013e, 27, Finished, Available, Finished, False)

Số dòng ban đầu: 5250
Số dòng sau khi deduplicate: 5000
Số dòng trùng bị loại bỏ: 250


In [26]:
# Kiểm tra các định dạng đang có trong cột order_date

from pyspark.sql import functions as F

display(
    df_sales_deduplicated
    .select("order_date")
    .where(F.col("order_date").isNotNull())
    .distinct()
    .limit(30)
)

StatementMeta(, 07588210-2439-412a-914d-e88280c6013e, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5de45e71-be9b-4079-8880-a26261058fd1)

In [27]:
# Parse 2 định dạng

df_sales = df_sales_deduplicated.withColumn(
    "order_date_parsed",
    F.coalesce(
        F.to_timestamp(F.trim(F.col("order_date")), "yyyy-MM-dd'T'HH:mm:ss.SSSSSS"),
        F.to_timestamp(F.trim(F.col("order_date")), "dd/MM/yyyy HH:mm")
    )
)

display(df_sales.select("order_date_parsed").limit(10))

StatementMeta(, 07588210-2439-412a-914d-e88280c6013e, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cb88b61b-db11-4394-b9bd-4d89af7aabd5)

In [29]:
# Tách customer_info thành customer_name, customer_email và customer_phone
from pyspark.sql.types import StructType, StructField, StringType

# Khai báo schema cho cột customer_info dạng JSON
customer_schema = StructType([
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True)
])

# Parse customer_info từ chuỗi JSON thành struct
df_sales = df_sales.withColumn(
    "customer_info_struct",
    F.from_json(F.col("customer_info"), customer_schema)
)

# Tách các field trong struct thành các cột riêng
df_sales = (
    df_sales
    .withColumn("customer_name", F.col("customer_info_struct.name"))
    .withColumn("customer_email", F.col("customer_info_struct.email"))
    .withColumn("customer_phone", F.col("customer_info_struct.phone"))
    .drop("customer_info_struct")
)

# Kiểm tra kết quả
display(
    df_sales
    .select("order_id", "customer_name", "customer_email", "customer_phone")
    .limit(30)
)

StatementMeta(, 07588210-2439-412a-914d-e88280c6013e, 31, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c0dc2be3-c2ad-40c0-a34e-2412551fe245)

In [31]:
# Chuẩn hóa customer_age
df_sales = df_sales.withColumn(
    "customer_age_clean",
    F.when(
        F.trim(F.col("customer_age").cast("string")).cast("int").between(0, 120),
        F.trim(F.col("customer_age").cast("string")).cast("int")
    ).otherwise(F.lit(None).cast("int"))
)

display(
    df_sales
    .select("order_id", "customer_age", "customer_age_clean")
    .limit(30)
)

StatementMeta(, 07588210-2439-412a-914d-e88280c6013e, 33, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3499b11a-53d4-4d03-b3ab-65f1c061b263)

In [33]:
# Chuẩn hóa payment_method
from pyspark.sql import functions as F

df_sales = df_sales.withColumn(
    "payment_method_key",
    F.upper(
        F.regexp_replace(
            F.trim(F.col("payment_method").cast("string")),
            "[^a-zA-Z]",
            ""
        )
    )
)

df_sales = (
    df_sales
    .withColumn(
        "payment_method_clean",
        F.when(F.col("payment_method_key") == "CREDITCARD", F.lit("Credit Card"))
         .otherwise(F.col("payment_method"))
    )
    .drop("payment_method_key")
)

display(
    df_sales
    .select("order_id", "payment_method", "payment_method_clean")
    .limit(10)
)

StatementMeta(, 07588210-2439-412a-914d-e88280c6013e, 35, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d2708e3f-c694-4261-b8fe-ffa41f5fb75e)

In [35]:
# Chuẩn hóa total_amount
from pyspark.sql.types import DecimalType

df_sales = df_sales.withColumn(
    "total_amount_clean",
    F.regexp_replace(
        F.trim(F.col("total_amount").cast("string")),
        "\\$",
        ""
    ).cast(DecimalType(12, 2))
)

display(
    df_sales
    .select("order_id", "total_amount", "total_amount_clean")
    .limit(30)
)

StatementMeta(, 07588210-2439-412a-914d-e88280c6013e, 37, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6588771b-d7ac-405e-b68b-0b1802d2ddbe)

In [37]:
# Phân loại dữ liệu
from pyspark.sql import functions as F

# Tạo điều kiện xác định dòng dữ liệu sạch
clean_condition = (
    F.col("order_id").isNotNull() &
    F.col("order_date_parsed").isNotNull() &
    F.col("customer_name").isNotNull() &
    F.col("customer_email").isNotNull() &
    F.col("customer_age_clean").isNotNull() &
    F.col("payment_method_clean").isNotNull() &
    F.col("shipping_cost").isNotNull() &
    (F.col("shipping_cost") >= 0) &
    F.col("total_amount_clean").isNotNull() &
    (F.col("total_amount_clean") >= 0)
)

# Dữ liệu sạch
df_sales_clean = df_sales.filter(clean_condition)

# Dữ liệu bị cách ly
df_sales_quarantine = (
    df_sales
    .filter(~clean_condition)
    .withColumn(
        "quarantine_reason",
        F.concat_ws(
            "; ",
            F.when(F.col("order_id").isNull(), "missing_order_id"),
            F.when(F.col("order_date_parsed").isNull(), "invalid_order_date"),
            F.when(F.col("customer_name").isNull(), "missing_customer_name"),
            F.when(F.col("customer_email").isNull(), "missing_customer_email"),
            F.when(F.col("customer_age_clean").isNull(), "invalid_customer_age"),
            F.when(F.col("total_amount_clean").isNull(), "invalid_total_amount"),
            F.when(F.col("shipping_cost") < 0, "negative_shipping_cost"),
            F.when(F.col("total_amount_clean") < 0, "negative_total_amount")
        )
    )
)

StatementMeta(, 07588210-2439-412a-914d-e88280c6013e, 39, Finished, Available, Finished, False)

In [38]:
print(f"Số dòng sạch: {df_sales_clean.count()}")
print(f"Số dòng bị cách ly: {df_sales_quarantine.count()}")

display(df_sales_clean.limit(30))
display(df_sales_quarantine.select(
    "order_id",
    "order_date",
    "customer_age",
    "shipping_cost",
    "total_amount",
    "quarantine_reason"
).limit(30))

StatementMeta(, 07588210-2439-412a-914d-e88280c6013e, 40, Finished, Available, Finished, False)

Số dòng sạch: 4441
Số dòng bị cách ly: 559


SynapseWidget(Synapse.DataFrame, 7b93fe17-73ce-4f43-a53b-697a27a251ce)

SynapseWidget(Synapse.DataFrame, 28b065a7-bc9e-45a6-9682-2d761145460c)

In [41]:
# Tạo DataFrame Silver chứa các cột đã clean
silver_columns = [
    F.col("order_id"),
    F.col("order_date_parsed").alias("order_date"),
    F.col("customer_name"),
    F.col("customer_email"),
    F.col("customer_phone"),
    F.col("customer_age_clean").alias("customer_age"),
    F.col("location"),
    F.col("device_type"),
    F.col("items"),
    F.col("payment_method_clean").alias("payment_method"),
    F.col("currency"),
    F.col("discount_code"),
    F.col("shipping_cost"),
    F.col("total_amount_clean").alias("total_amount"),
    F.col("order_status"),
    F.col("loyalty_points"),
    F.col("feedback_score").cast("integer"),
]

df_silver_sales = df_sales_clean.select(*silver_columns)
df_silver_sales.printSchema()

StatementMeta(, 07588210-2439-412a-914d-e88280c6013e, 43, Finished, Available, Finished, False)

root
 |-- order_id: string (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- customer_phone: string (nullable = true)
 |-- customer_age: integer (nullable = true)
 |-- location: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- items: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- discount_code: string (nullable = true)
 |-- shipping_cost: double (nullable = true)
 |-- total_amount: decimal(12,2) (nullable = true)
 |-- order_status: string (nullable = true)
 |-- loyalty_points: long (nullable = true)
 |-- feedback_score: integer (nullable = true)



In [43]:
# Lưu Delta Tables

# Silver Clean Table
df_silver_sales.write.format("delta").mode("overwrite").saveAsTable("silver_sales")

# Silver Quarantine Table
df_sales_quarantine.write.format("delta").mode("overwrite").saveAsTable("silver_sales_quarantine")

StatementMeta(, 07588210-2439-412a-914d-e88280c6013e, 45, Finished, Available, Finished, False)

In [44]:
spark.table("silver_sales").show(10)

StatementMeta(, 07588210-2439-412a-914d-e88280c6013e, 46, Finished, Available, Finished, False)

+----------+--------------------+------------------+--------------------+--------------------+------------+---------------+-----------+--------------------+--------------+--------+-------------+-------------+------------+------------+--------------+--------------+
|  order_id|          order_date|     customer_name|      customer_email|      customer_phone|customer_age|       location|device_type|               items|payment_method|currency|discount_code|shipping_cost|total_amount|order_status|loyalty_points|feedback_score|
+----------+--------------------+------------------+--------------------+--------------------+------------+---------------+-----------+--------------------+--------------+--------+-------------+-------------+------------+------------+--------------+--------------+
|TXN-104974|2024-11-20 14:09:...|        Shawn Soto|taylorbrandon@exa...| (255)897-8613x17972|          46|     Port Julia|    Desktop|[{"product_id": "...|        PayPal|     USD|         Eb10|         8.